### 💡 Data Quality : 데이터 품질 검사
**데이터 품질 검사**란 데이터 분석을 시작하기 전에 데이터가 올바른 상태인지 확인하는 과정이다.

예륻들어 나이가 비어있거나, 동일한 데이터가 반복되거나, 나이가 -5 요금이 -10 같은 비정상적인 데이터가 들어있으면 나중에 분석할 때 잘못된 값으로 나올 수 있다.

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("./datasets/titanic.csv")

df.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


### 1. 결측치 분석
**결측치는 데이터에 값이 존재하지 않는 상태**이다. Pandas에서는 대부분 `NaN`로 표현하는 경우가 많다.

In [3]:
# 결측치 확인
# 컬럼별 결측치 갯수
print(df.isnull().sum())

# 컬럼별 결측치 비율
print(df.isnull().mean() * 100)

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64
survived        0.000000
pclass          0.000000
sex             0.000000
age            19.865320
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.224467
class           0.000000
who             0.000000
adult_male      0.000000
deck           77.216611
embark_town     0.224467
alive           0.000000
alone           0.000000
dtype: float64


In [4]:
# 결측치가 있는 행 확인하기
print(df[df["age"].isnull()])

# 나이 또는 요금이 비어있는 승객 확인
print(df[df[["age", "fare"]].isnull().any(axis=1)])

     survived  pclass     sex  age  sibsp  parch     fare embarked   class  \
5           0       3    male  NaN      0      0   8.4583        Q   Third   
17          1       2    male  NaN      0      0  13.0000        S  Second   
19          1       3  female  NaN      0      0   7.2250        C   Third   
26          0       3    male  NaN      0      0   7.2250        C   Third   
28          1       3  female  NaN      0      0   7.8792        Q   Third   
..        ...     ...     ...  ...    ...    ...      ...      ...     ...   
859         0       3    male  NaN      0      0   7.2292        C   Third   
863         0       3  female  NaN      8      2  69.5500        S   Third   
868         0       3    male  NaN      0      0   9.5000        S   Third   
878         0       3    male  NaN      0      0   7.8958        S   Third   
888         0       3  female  NaN      1      2  23.4500        S   Third   

       who  adult_male deck  embark_town alive  alone  
5      

결측치를 처리하는 방법은 행삭제, 컬럼삭제, 평균대체 등 방법이 여러가지이다. 하지만 결측치가 있다고 무조건 삭제하거나 평균으로 대체하면 안된다는 것이다.<br/>
그 이유는 타이타닉의 age는 나이가 비어있는 승객을 전부 제거하게되면 분석 대상이 줄어들고, 반대로 평균으로 채우면 실제 나이 분포와 차이가 생길 수 있다.

### 2. 중복 데이터 분석
중복 데이터는 동일한 관측치가 여러 번 기록된 경우이다.

In [5]:
# 중복 행 개수 확인
# 완전히 동일한 행의 개수
print(df.duplicated().sum())

# 중복된 행 확인
print(df[df.duplicated()])

# 특정 컬럼 기준으로 중복된 행 확인
print(df[df.duplicated(subset=["sex", "age", "fare"])])

107
     survived  pclass     sex   age  sibsp  parch     fare embarked   class  \
47          1       3  female   NaN      0      0   7.7500        Q   Third   
76          0       3    male   NaN      0      0   7.8958        S   Third   
77          0       3    male   NaN      0      0   8.0500        S   Third   
87          0       3    male   NaN      0      0   8.0500        S   Third   
95          0       3    male   NaN      0      0   8.0500        S   Third   
..        ...     ...     ...   ...    ...    ...      ...      ...     ...   
870         0       3    male  26.0      0      0   7.8958        S   Third   
877         0       3    male  19.0      0      0   7.8958        S   Third   
878         0       3    male   NaN      0      0   7.8958        S   Third   
884         0       3    male  25.0      0      0   7.0500        S   Third   
886         0       2    male  27.0      0      0  13.0000        S  Second   

       who  adult_male deck  embark_town alive 

### 3. 유효하지 않은 값 검사
데이터가 비어있지 않더라도 논리적으로 잘못된 값들이 있을 수 있다. (ex. 나이가 -01살, 요금이 -100, 객실등급이 5)

In [6]:
# 수치형 데이터 검사
# 음수 나이 확인
print(df[df["age"] < 0])

# 음수 요금 확인
print(df[df["fare"] < 0])

# 객실 등급이 1, 2, 3 이외의 값인지 확인
print(df[~df["pclass"].isin([1, 2, 3])])

# 허용된 성별말고 이외의 값이 있는지 확인
print(df[~df["sex"].isin(["male", "female"])])

Empty DataFrame
Columns: [survived, pclass, sex, age, sibsp, parch, fare, embarked, class, who, adult_male, deck, embark_town, alive, alone]
Index: []
Empty DataFrame
Columns: [survived, pclass, sex, age, sibsp, parch, fare, embarked, class, who, adult_male, deck, embark_town, alive, alone]
Index: []
Empty DataFrame
Columns: [survived, pclass, sex, age, sibsp, parch, fare, embarked, class, who, adult_male, deck, embark_town, alive, alone]
Index: []
Empty DataFrame
Columns: [survived, pclass, sex, age, sibsp, parch, fare, embarked, class, who, adult_male, deck, embark_town, alive, alone]
Index: []


### 4. 일관성 없는 데이터 검사
예를들어 성별이 MALE, male, Male 등 일관성이 없다면 컴퓨터는 서로 다른 문자열로 인식하게 된다.

In [7]:
# 성별 데이터의 종류 확인
print(df["sex"].unique())

# 탑승 항구 데이터의 종류 확인
print(df["embarked"].unique())

<ArrowStringArray>
['male', 'female']
Length: 2, dtype: str
<ArrowStringArray>
['S', 'C', 'Q', nan]
Length: 4, dtype: str


### 데이터 품질 검사 종합
1. **결측치** : 비어있는 데이터가 있는가?
2. **중복** : 동일한 관측치가 있는가?
3. **유효성** : 허용 범위를 벗어난 값이 있는가?
4. **일관성** : 같은 의미가 동일한 형식으로 표현되는가?

---
### 실습문제

In [8]:
# A. 결측치 검사
# 1. 전체 컬럼의 결측치 개수를 출력하기
print(df.isnull().sum())

# 2. 전체 컬럼의 결측치 비율을 백분율로 출력하기
print(df.isnull().sum() / len(df) * 100)

# 3. age에 결측치가 있는 행만 출력하기
print(df[df["age"].isnull()])

# 4. age 또는 embarked에 결측치가 있는 행만 출력하기
print(df[df[["age", "embarked"]].isnull().any(axis=1)])

# 5. age와 fare 두 컬럼 모두 결측치인 행 출력하기
print(df[df[["age", "fare"]].isnull().all(axis=1)])

# 6. age와 fare 두 컬럼 중 하나라도 결측치인 행의 개수 출력하기
print(df[["age", "fare"]].isnull().any(axis=1).sum())

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64
survived        0.000000
pclass          0.000000
sex             0.000000
age            19.865320
sibsp           0.000000
parch           0.000000
fare            0.000000
embarked        0.224467
class           0.000000
who             0.000000
adult_male      0.000000
deck           77.216611
embark_town     0.224467
alive           0.000000
alone           0.000000
dtype: float64
     survived  pclass     sex  age  sibsp  parch     fare embarked   class  \
5           0       3    male  NaN      0      0   8.4583        Q   Third   
17          1       2    male  NaN      0      0  13.0000        S  Second   
19          1       3  female  NaN      0      0   7.2250        C   Third   


In [14]:
# B. 중복 데이터 검사
# 7. 전체 데이터에서 완전히 동일한 행의 개수 확인하기
print(df.duplicated().sum())

# 8. sex, age, fare를 기준으로 중복된 행 확인하기
print(df[df.duplicated(subset=["sex", "age", "fare"])])

107
     survived  pclass     sex   age  sibsp  parch     fare embarked   class  \
42          0       3    male   NaN      0      0   7.8958        C   Third   
47          1       3  female   NaN      0      0   7.7500        Q   Third   
76          0       3    male   NaN      0      0   7.8958        S   Third   
77          0       3    male   NaN      0      0   8.0500        S   Third   
87          0       3    male   NaN      0      0   8.0500        S   Third   
..        ...     ...     ...   ...    ...    ...      ...      ...     ...   
877         0       3    male  19.0      0      0   7.8958        S   Third   
878         0       3    male   NaN      0      0   7.8958        S   Third   
881         0       3    male  33.0      0      0   7.8958        S   Third   
884         0       3    male  25.0      0      0   7.0500        S   Third   
886         0       2    male  27.0      0      0  13.0000        S  Second   

       who  adult_male deck  embark_town alive 

In [18]:
# C. 유효성 검사
# 9. 나이가 음수인 승객 확인하기
print(df[df["age"] < 0])

# 10. 요금이 음수인 승객 확인하기
print(df[df["fare"] < 0])

# 11. 객실 등급이 1, 2, 3 이외의 값인 승객 확인하기
print(df[~df["pclass"].isin([1, 2, 3])])

# 12. 성별이 male, female 이외의 값인 승객 확인하기
print(df[~df["sex"].isin(["male", "female"])])

Empty DataFrame
Columns: [survived, pclass, sex, age, sibsp, parch, fare, embarked, class, who, adult_male, deck, embark_town, alive, alone]
Index: []
Empty DataFrame
Columns: [survived, pclass, sex, age, sibsp, parch, fare, embarked, class, who, adult_male, deck, embark_town, alive, alone]
Index: []
Empty DataFrame
Columns: [survived, pclass, sex, age, sibsp, parch, fare, embarked, class, who, adult_male, deck, embark_town, alive, alone]
Index: []
Empty DataFrame
Columns: [survived, pclass, sex, age, sibsp, parch, fare, embarked, class, who, adult_male, deck, embark_town, alive, alone]
Index: []


D. 분석 및 판단 문제

13. 결측치가 있는 행을 무조건 삭제하면 안 되는 이유는 무엇일까?
    > 만약 결측치가 있는 행이 한두개가 아니라 매우 많은 행이라면 해당 행을 삭제했을 때 데이터분석에 어려움이 생길 수 있기 때문이다.

14. 나이와 요금이 동일한 승객 두 명은 반드시 중복 데이터일까?
    > 실제로 나이도 같고 요금이 같은 다른 두 승객일 수 있으니 무조건 중복데이터는 아니다.

15. 요금이 다른 승객보다 매우 높은 경우, 무조건 잘못된 데이터라고 판단해도 될까?
    > 실제로 그 값을 지불한 승객일 수 있으니 무조건 잘못된 데이터는 아니다.

16. 결측치가 많은 컬럼은 무조건 삭제해야 할까?
    > 결측치가 많은 컬럼을 지웠을 때 실제 데이터 분석에 어려움이 있을 수 있으니 무조건 지우면 안된다.